# Baselines A and B on HMDB51 (Kaggle)

Fifth Kaggle session, second domain. All runs use the XOV HMDB51 test list: 1,530 videos, 51
classes, 33 familiar / 18 unfamiliar. Every setting is the one already used on UCF101, and
nothing is tuned.

1. **Gate:** frozen CLIP ViT-B/32 on 2 videos per class (102 videos). The full runs start only
   if this small run succeeds.
2. **Baseline A, frozen CLIP:** ViT-B/32 and ViT-B/16, each with 1 prompt template and with 28
   templates, on the common protocol (as on UCF101 in session 1).
3. **Baseline B, Open-VCLIP adapter:** the native configuration (8×3×1) and the common protocol
   (16×1×1).
4. **Official Open-VCLIP test script on the same 1,530 videos**, then `compare-native`. The
   official HMDB51 script (`script/testing/temporal_test_b16/hmdb.sh`) decodes with
   **torchvision**, not PyAV. The pip wheel of torchvision 0.12 has no video reader, so this
   stage builds Open-VCLIP's INSTALL.md environment with conda. The conda torchvision build is
   compiled with FFmpeg.

**Before running**

1. Settings → Accelerator: **GPU T4**. Settings → Internet: **On**.
2. **Add input:** dataset **kodurujagruthaaditya/hmdb51** (the original `hmdb51_org` AVI files,
   2.1 GB). The official download link of the Serre lab no longer works, so there is no automatic
   download.
3. Save Version → **Save & Run All (Commit)**. A failing stage is recorded and the others still
   run; the last cell lists every stage. Download **`ovar_outputs_5.zip`** from the Output tab.

Rough time: setup 10 min, frozen CLIP about 40 min, Open-VCLIP adapter about 40 min, official
environment and test about 35 min. About 2 hours in total.

In [ ]:
import os, subprocess, sys, json, hashlib, shutil
from pathlib import Path

REPO = "eylulpelinkilic/generalized-category-discovery"
REF = "main"            # replace with a commit SHA to pin the code version
WORK = Path("/kaggle/working")
CODE = WORK / "generalized-category-discovery"
PROJECT = CODE / "action_recognition"
RUN_CLIP = True         # 2. frozen CLIP, four configurations
RUN_OVC = True          # 3. Open-VCLIP adapter, native and common
RUN_OFFICIAL = True     # 4. official Open-VCLIP test script (conda environment) + comparison
ZIP = "ovar_outputs_5.zip"
STATUS = {}


def run(cmd, cwd=None, allow_fail=False, log=None):
    """Run a shell command, stream (and optionally save) its output, raise on failure."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (PROJECT if PROJECT.exists() else WORK),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    sink = open(log, "a") if log else None
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-15:]
        if sink:
            sink.write(line)
    code = p.wait()
    if sink:
        sink.close()
    if code and not allow_fail:
        last = "\n".join(tail)
        raise RuntimeError(f"command failed with exit code {code}: {cmd}\n--- last output lines ---\n{last}")
    return code


def step(name, fn, needs=(), enabled=True):
    """Run one stage; record a failure and continue, so independent later stages still run."""
    missing = [n for n in needs if STATUS.get(n) != "ok"]
    if not enabled:
        STATUS[name] = "not requested"
        return
    if missing:
        STATUS[name] = f"skipped: needs {', '.join(missing)}"
        print(f"-- {name} skipped (needs {missing})")
        return
    try:
        fn()
        STATUS[name] = "ok"
    except Exception as exc:
        STATUS[name] = f"failed: {exc}"
        print(f"!! {name} failed: {exc}")


def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

## 1. Code: clone the repository

In [ ]:
if subprocess.run(["git", "ls-remote", f"https://github.com/{REPO}", REF],
                  capture_output=True, text=True).returncode != 0:
    raise RuntimeError("Cannot reach GitHub. Turn Settings -> Internet on (phone-verified account), "
                       "then restart the session.")

if not CODE.exists():
    CODE.mkdir(parents=True)
    run("git init -q && git remote add origin https://github.com/" + REPO + ".git", cwd=CODE)
run(f"git fetch -q --depth 1 origin {REF} && git checkout -q FETCH_HEAD", cwd=CODE)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
print("code commit:", COMMIT)
assert (PROJECT / "kaggle" / "ovar_hmdb51_baselines.ipynb").exists(), \
    "the cloned code predates this notebook: push the latest commit to GitHub (or set REF)"

## 2. Environment (this repository)

In [ ]:
run("pip install -q open_clip_torch ftfy regex")
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg")
run("ffmpeg -version | head -1")

import torch, open_clip
print("python", sys.version.split()[0], "| torch", torch.__version__, "| open_clip", open_clip.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Settings -> Accelerator: GPU"

run("python -m ovar fetch-repos")
run("python -m unittest discover -s tests -t .")

## 3. Data: the attached HMDB51 copy

The notebook looks for the class folders under `/kaggle/input` and detects the layout:
`brush_hair/<clip>.avi` (single class folder) or `brush_hair/brush_hair/<clip>.avi` (what extracting
the per-class RARs gives). The original release has 6,766 clips. Validation then checks that
every one of the 1,530 listed test files exists, and decodes 100 of them.

In [ ]:
def find_hmdb():
    found = []
    for depth in range(5):
        for p in Path("/kaggle/input").glob("*/" * depth + "brush_hair"):
            if any(p.glob("*.avi")):
                found.append((p.parent, True))
            elif any(p.glob("brush_hair/*.avi")):
                found.append((p.parent, False))
    return sorted(found, key=lambda x: len(x[0].parts))


candidates = find_hmdb()
if not candidates:
    raise RuntimeError("no HMDB51 class folders under /kaggle/input: add the dataset "
                       "kodurujagruthaaditya/hmdb51 (Add Input), then run again")
HMDB_ROOT, SINGLE_FOLDER = candidates[0]
DATA_SOURCE = {"source": "kaggle input", "path": str(HMDB_ROOT), "single_class_folder": SINGLE_FOLDER,
               "other_copies": [str(p) for p, _ in candidates[1:]],
               "n_avi": sum(1 for _ in HMDB_ROOT.rglob("*.avi")),
               "n_class_folders": sum(1 for p in HMDB_ROOT.iterdir() if p.is_dir())}
print(json.dumps(DATA_SOURCE, indent=2))
(PROJECT / "runs").mkdir(exist_ok=True)
(PROJECT / "runs" / "DATA_SOURCE.json").write_text(json.dumps(DATA_SOURCE, indent=2) + "\n")

CKPT_DIR = Path("/tmp/checkpoints")              # outside /kaggle/working: not part of the outputs
OVC_CKPT = CKPT_DIR / "openvclip-b16" / "swa_2_22.pth"
CLIP_JIT = Path.home() / ".cache" / "clip" / "ViT-B-16.pt"   # where the official code looks for it
(PROJECT / "configs" / "paths.yaml").write_text(
    "dataset_roots:\n"
    f"  hmdb51: {HMDB_ROOT}\n"
    f"hmdb51_single_class_folder: {'true' if SINGLE_FOLDER else 'false'}\n"
    "checkpoints:\n"
    f"  openvclip_vitb16: {OVC_CKPT}\n"
    f"  clip_vitb16_openai_pt: {CLIP_JIT}\n"
    "cache_dir: cache\n"
    "runs_dir: runs\n")
print((PROJECT / "configs" / "paths.yaml").read_text())
run("python -m ovar validate --datasets hmdb51 --check-files --check-decode --decode-limit 100")

## 4. Weights

* open_clip ViT-B/32 and ViT-B/16 for frozen CLIP (same files as session 1).
* Open-VCLIP `swa_2_22.pth`: its SHA-256 must equal sessions 3–4's (docs/backbones.md: `49a5bf45…`).
* Original OpenAI `ViT-B-16.pt`, used to build Open-VCLIP and for weight patching. Its URL contains
  its SHA-256, which is verified.

In [ ]:
OVC_URL = ("https://drive.usercontent.google.com/download?id=1QLdkVYPP_AjFJgB9mGNLhppqvHFWkT4x"
           "&export=download&confirm=t")
OVC_BYTES = 598_587_149
OVC_SHA = "49a5bf452d044f18e8a3c2dff63d46986aeb082be22efe74b70df28cfe08e570"   # sessions 3-4
CLIP_JIT_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"
CLIP_JIT_URL = f"https://openaipublic.azureedge.net/clip/models/{CLIP_JIT_SHA}/ViT-B-16.pt"
CLIP_JIT_BYTES = 350_837_078
DOWNLOADS = {}


def fetch(url, dest, nbytes):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not (dest.exists() and dest.stat().st_size == nbytes):
        run(f"curl -sS -L --fail --retry 5 -C - -o '{dest}' '{url}'")
    size = dest.stat().st_size
    if size != nbytes:
        raise RuntimeError(f"{dest.name}: got {size} bytes, expected {nbytes}. If Google Drive refuses "
                           "(download quota), download the file in a browser, upload it as a private "
                           f"Kaggle dataset, and copy it to {dest}.")
    return {"url": url, "bytes": size, "sha256": sha256(dest)}


def clip_weights():
    run("python -m ovar fetch-clip --arch ViT-B-32 --yes")
    run("python -m ovar fetch-clip --arch ViT-B-16 --yes")


def ovc_weights():
    DOWNLOADS["openvclip_vitb16"] = fetch(OVC_URL, OVC_CKPT, OVC_BYTES)
    DOWNLOADS["clip_vitb16_openai_pt"] = fetch(CLIP_JIT_URL, CLIP_JIT, CLIP_JIT_BYTES)
    print(json.dumps(DOWNLOADS, indent=2))
    (PROJECT / "runs" / "DOWNLOADS.json").write_text(json.dumps(DOWNLOADS, indent=2) + "\n")
    if DOWNLOADS["clip_vitb16_openai_pt"]["sha256"] != CLIP_JIT_SHA:
        raise RuntimeError("ViT-B-16.pt SHA-256 does not match its official URL")
    if DOWNLOADS["openvclip_vitb16"]["sha256"] != OVC_SHA:
        raise RuntimeError("swa_2_22.pth differs from the checkpoint checked in sessions 3-4")


step("clip_weights", clip_weights)
step("ovc_weights", ovc_weights, enabled=RUN_OVC or RUN_OFFICIAL)

## 5. Gate: small run on 2 videos per class

New dataset on real videos: first a seeded 102-video run with frozen CLIP ViT-B/32. The full runs
below need it to succeed.

In [ ]:
def gate():
    run("python -m ovar infer clip_vitb32__common_16x1x1 --per-class 2 --set 'datasets=[hmdb51]' "
        "--variant hmdb51_subset2 --n-boot 200")
    failed = json.loads((PROJECT / "runs" / "clip_vitb32__common_16x1x1__hmdb51_subset2" / "run.json")
                        .read_text())["datasets"]["hmdb51"]["failed"]
    if failed:
        raise RuntimeError(f"{failed} of 102 videos failed to decode: see failures.csv before the full runs")


step("gate", gate, needs=["clip_weights"])

## 6. Baseline A: frozen CLIP on the full HMDB51 test list

The 28-template runs reuse the cached video embeddings of the 1-template runs.

In [ ]:
CLIP_EXPERIMENTS = ["clip_vitb32__common_16x1x1", "clip_vitb32__common_16x1x1_ens28",
                    "clip_vitb16__common_16x1x1", "clip_vitb16__common_16x1x1_ens28"]
for exp in CLIP_EXPERIMENTS:
    step(exp, lambda exp=exp: run(f"python -m ovar infer {exp} --set 'datasets=[hmdb51]' --variant hmdb51"),
         needs=["gate"], enabled=RUN_CLIP)

## 7. Baseline B: Open-VCLIP adapter, native configuration and common protocol

The checkpoint is a pickle, hence `model.trust_pickle=true`. It comes from the official Drive
folder, and its SHA-256 equals sessions 3–4's.

In [ ]:
NATIVE_RUN = PROJECT / "runs" / "openvclip_vitb16__openvclip_native_8x3x1__hmdb51"
for exp in ["openvclip_vitb16__openvclip_native_8x3x1", "openvclip_vitb16__common_16x1x1"]:
    step(exp, lambda exp=exp: run(f"python -m ovar infer {exp} --set 'datasets=[hmdb51]' "
                                  "--set model.trust_pickle=true --variant hmdb51"),
         needs=["gate", "ovc_weights"], enabled=RUN_OVC)

## 8. Official Open-VCLIP environment, as in its INSTALL.md (conda)

`conda create python=3.8.13` and `conda install pytorch==1.11.0 torchvision==0.12.0
torchaudio==0.11.0 cudatoolkit=11.3 -c pytorch`, with micromamba as the conda client. The CUDA
builds are pinned by build string. The conda torchvision depends on FFmpeg and includes the video
reader that the official HMDB51 setting needs (`DATA.DECODING_BACKEND torchvision`). The pip
wheel used for UCF101 does not.

The remaining packages, the detectron2 import-only stub and the `PYTHONPATH` setup are as in
sessions 3–4. A probe decodes one HMDB51 test video with torchvision's video reader before the
test runs.

In [ ]:
OVC = PROJECT / "third_party" / "Open-VCLIP"
MAMBA = Path("/tmp/micromamba/bin/micromamba")
ENV = Path("/tmp/ovc-conda")
PY38 = ENV / "bin" / "python"
STUBS = Path("/tmp/ovc-stubs")
PYTORCHVIDEO_COMMIT = "eb04d1b21e08cfd0713164c0907aeb4c98fd83af"   # main on 2024-02-19
CONSTRAINTS = Path("/tmp/ovc-constraints.txt")
OVC_PATH = f"PYTHONPATH={STUBS}:{OVC}"
# mkl < 2024.1: newer mkl breaks importing torch 1.11 ("undefined symbol: iJIT_NotifyEvent")
CONDA_SPECS = ["python=3.8.13", "pip", "numpy=1.23.5", "mkl<2024.1",
               "pytorch=1.11.0=py3.8_cuda11.3_cudnn8.2.0_0", "torchvision=0.12.0=py38_cu113",
               "torchaudio=0.11.0=py38_cu113", "cudatoolkit=11.3"]

STUB_SOURCE = (
    '"""Import-only stand-in for detectron2 (see the notebook). Using anything from it raises."""\n'
    "class _Unavailable:\n"
    "    def __init__(self, *args, **kwargs):\n"
    "        raise RuntimeError(\"detectron2 is stubbed: this code path needs the real package\")\n\n\n"
    "def __getattr__(name):\n"
    "    return _Unavailable\n")

PROBE = (
    "import sys, numpy, torch, torchvision\n"
    "from torchvision.io import _video_opt\n"
    "print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'numpy', numpy.__version__)\n"
    "print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0)); torch.zeros(1).cuda()\n"
    "assert _video_opt._HAS_VIDEO_OPT, 'torchvision has no video reader'\n"
    "data = torch.from_numpy(numpy.fromfile(sys.argv[1], dtype=numpy.uint8))\n"
    "frames, _ = torchvision.io._read_video_from_memory(data)\n"
    "print('decoded', tuple(frames.shape), 'from', sys.argv[1])\n"
    "assert frames.shape[0] > 0\n")


def official_env():
    run(f"python -m ovar export-native --backbone openvclip --datasets hmdb51 --out {EXPORT}")
    run(f"head -2 {EXPORT / 'hmdb51' / 'test.csv'}")
    if not MAMBA.exists():
        MAMBA.parent.parent.mkdir(parents=True, exist_ok=True)
        run(f"curl -sS -L --fail --retry 5 https://micro.mamba.pm/api/micromamba/linux-64/latest "
            f"| tar -xj -C {MAMBA.parent.parent} bin/micromamba")
    if not PY38.exists():
        specs = " ".join(f"'{s}'" for s in CONDA_SPECS)
        run(f"MAMBA_ROOT_PREFIX=/tmp/micromamba/root {MAMBA} create -y -q -p {ENV} "
            f"--override-channels -c pytorch -c conda-forge {specs}")
    CONSTRAINTS.write_text("torch==1.11.0\ntorchvision==0.12.0\nnumpy==1.23.5\nsetuptools==59.5.0\n"
                           "av==12.3.0\nprotobuf==3.20.3\n")
    run(f"{PY38} -m pip install -q -c {CONSTRAINTS} setuptools")
    run(f"{PY38} -m pip install -q -c {CONSTRAINTS} scipy pandas scikit-learn ftfy regex simplejson psutil "
        f"opencv-python-headless tensorboard iopath fvcore av matplotlib tqdm termcolor yacs pyyaml "
        f"'pytorchvideo @ git+https://github.com/facebookresearch/pytorchvideo.git@{PYTORCHVIDEO_COMMIT}'")
    for rel in ["detectron2/__init__.py", "detectron2/layers.py", "detectron2/config.py",
                "detectron2/engine.py", "detectron2/model_zoo.py", "detectron2/utils/__init__.py",
                "detectron2/utils/visualizer.py"]:
        (STUBS / rel).parent.mkdir(parents=True, exist_ok=True)
        (STUBS / rel).write_text(STUB_SOURCE)
    probe = Path("/tmp/ovc-probe.py")
    probe.write_text(PROBE)
    sample = HMDB_ROOT / (EXPORT / "hmdb51" / "test.csv").read_text().split(",")[0]
    run(f"{PY38} {probe} '{sample}'")  # HMDB51 names contain & ( ) ! but no quotes
    run(f"{OVC_PATH} {PY38} -c \"import sys; sys.path.insert(0, 'tools'); import run_net; "
        f"print('official code imports')\"", cwd=OVC)
    run(f"{PY38} -m pip freeze > {PROJECT / 'runs' / 'official_env_freeze.txt'}")
    run(f"MAMBA_ROOT_PREFIX=/tmp/micromamba/root {MAMBA} list -p {ENV} > {PROJECT / 'runs' / 'official_env_conda.txt'}")


CHECK = PROJECT / "runs" / "native_check"
EXPORT = CHECK / "export_ovc_hmdb51"
OFFICIAL_OUT = CHECK / "official_ovc_hmdb51"
step("official_env", official_env, needs=["ovc_weights"], enabled=RUN_OFFICIAL)

## 9. Official test script on the same 1,530 videos, then the comparison

The settings are those of `script/testing/temporal_test_b16/hmdb.sh`: torchvision decoding,
51 classes, 3 views × 1 crop, weight patching 0.5. Only the paths, the batch size, the GPU count
and the loader workers differ. The class texts are ours, in XOV's label order (Open-VCLIP's own
HMDB51 index order differs for ten classes).

In [ ]:
def official_hmdb():
    OFFICIAL_OUT.mkdir(parents=True, exist_ok=True)
    run(f"{OVC_PATH} {PY38} -W ignore -u tools/run_net.py "
        "--cfg configs/Kinetics/TemporalCLIP_vitb16_8x16_STAdapter.yaml --opts "
        f"DATA.PATH_TO_DATA_DIR {EXPORT / 'hmdb51'} DATA.PATH_PREFIX {HMDB_ROOT} DATA.PATH_LABEL_SEPARATOR , "
        f"DATA.INDEX_LABEL_MAPPING_FILE {EXPORT / 'hmdb51' / 'hmdb51-index2cls.json'} "
        f"TRAIN.ENABLE False OUTPUT_DIR {OFFICIAL_OUT} TEST.BATCH_SIZE 8 NUM_GPUS 1 "
        "DATA.DECODING_BACKEND torchvision MODEL.NUM_CLASSES 51 DATA_LOADER.NUM_WORKERS 4 "
        f"TEST.CUSTOM_LOAD True TEST.CUSTOM_LOAD_FILE {OVC_CKPT} TEST.SAVE_RESULTS_PATH preds.pkl "
        "TEST.NUM_ENSEMBLE_VIEWS 3 TEST.NUM_SPATIAL_CROPS 1 "
        f"TEST.PATCHING_MODEL True TEST.PATCHING_RATIO 0.5 TEST.CLIP_ORI_PATH {CLIP_JIT}",
        cwd=OVC, log=OFFICIAL_OUT / "console.log")
    assert (OFFICIAL_OUT / "preds.pkl").exists(), "official script finished without writing preds.pkl"


def compare_hmdb():
    run(f"python -m ovar compare-native --official {OFFICIAL_OUT / 'preds.pkl'} --run-dir {NATIVE_RUN} "
        f"--dataset hmdb51 --trust-pickle --out {CHECK / 'compare_hmdb51_full.json'}")


step("official_hmdb", official_hmdb, needs=["official_env"], enabled=RUN_OFFICIAL)
step("compare_hmdb", compare_hmdb, needs=["official_hmdb", "openvclip_vitb16__openvclip_native_8x3x1"],
     enabled=RUN_OFFICIAL)

## 10. Reports and outputs

In [ ]:
step("report", lambda: run("python -m ovar report"))
if STATUS.get("report") == "ok":
    from IPython.display import Markdown, display
    display(Markdown((PROJECT / "reports" / "results.md").read_text()))
if (CHECK / "compare_hmdb51_full.json").exists():
    result = json.loads((CHECK / "compare_hmdb51_full.json").read_text())
    print("native check, HMDB51:", {k: v for k, v in result.items() if k != "disagreements"},
          f"| {len(result['disagreements'])} disagreeing videos")

(PROJECT / "runs" / "CODE_COMMIT.txt").write_text(COMMIT + "\n")
(PROJECT / "runs" / "NOTEBOOK_STATUS.json").write_text(json.dumps(STATUS, indent=2) + "\n")
run(f"rm -f {WORK}/{ZIP} && cd {PROJECT} && zip -qr {WORK}/{ZIP} runs reports configs/paths.yaml")
run(f"ls -lh {WORK}/{ZIP}")

In [ ]:
# Does not raise: an error here could cost the saved outputs of a committed run.
failed = {k: v for k, v in STATUS.items() if v not in ("ok", "not requested")}
print("=" * 70)
print("STAGES NOT COMPLETED - send the zip or this output:" if failed else "all requested stages completed")
for name, state in STATUS.items():
    print(f"{name:42s} {state}")
print("=" * 70)